# 02 Bronze - Retail

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Preserves source values and lineage while converting each dataset to Delta.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'retail':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

specs = {'customers': {'filename': 'customers.csv', 'primary_key': ['customer_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'loyalty_tier', 'type': 'STRING', 'required': True}, {'name': 'signup_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'products': {'filename': 'products.csv', 'primary_key': ['product_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'category', 'type': 'STRING', 'required': True}, {'name': 'brand_label', 'type': 'STRING', 'required': True}, {'name': 'unit_cost', 'type': 'DOUBLE', 'required': True}, {'name': 'list_price', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'orders': {'filename': 'orders.csv', 'primary_key': ['order_id'], 'foreign_keys': [['customer_id', 'customers', 'customer_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'order_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'channel', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'currency', 'type': 'STRING', 'required': True}, {'name': 'order_status', 'type': 'STRING', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'declared_total', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'order_items': {'filename': 'order_items.csv', 'primary_key': ['order_id', 'line_number'], 'foreign_keys': [['order_id', 'orders', 'order_id'], ['product_id', 'products', 'product_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'line_number', 'type': 'BIGINT', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'quantity', 'type': 'BIGINT', 'required': True}, {'name': 'unit_price', 'type': 'DOUBLE', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/products/"}
destinations = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/products/"}
landing_tables = {"customers": f"{participant_key}_retail_customers", "order_items": f"{participant_key}_retail_order_items", "orders": f"{participant_key}_retail_orders", "products": f"{participant_key}_retail_products"}

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_retail_customers (`participant_key` STRING, `source_row_id` STRING, `customer_id` STRING, `segment` STRING, `region` STRING, `loyalty_tier` STRING, `signup_date` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/customers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_retail_products (`participant_key` STRING, `source_row_id` STRING, `product_id` STRING, `category` STRING, `brand_label` STRING, `unit_cost` STRING, `list_price` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/products/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_retail_orders (`participant_key` STRING, `source_row_id` STRING, `order_id` STRING, `customer_id` STRING, `order_time` STRING, `channel` STRING, `region` STRING, `currency` STRING, `order_status` STRING, `discount_amount` STRING, `declared_total` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/orders/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_retail_order_items (`participant_key` STRING, `source_row_id` STRING, `order_id` STRING, `line_number` STRING, `product_id` STRING, `quantity` STRING, `unit_price` STRING, `discount_amount` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/order_items/'""")

for dataset, spec in specs.items():
    frame = (spark.table(f"aidp_lab.oci_landing.{landing_tables[dataset]}")
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp()))
    landing_count = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(destinations[dataset])
    bronze_count = spark.read.format("delta").load(destinations[dataset]).count()
    assert bronze_count == landing_count, f"Bronze count mismatch for {dataset}"
    print(f"Bronze {dataset}: {bronze_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_retail_customers (`participant_key` STRING, `source_row_id` STRING, `customer_id` STRING, `segment` STRING, `region` STRING, `loyalty_tier` STRING, `signup_date` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/customers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_retail_products (`participant_key` STRING, `source_row_id` STRING, `product_id` STRING, `category` STRING, `brand_label` STRING, `unit_cost` STRING, `list_price` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/products/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_retail_orders (`participant_key` STRING, `source_row_id` STRING, `order_id` STRING, `customer_id` STRING, `order_time` STRING, `channel` STRING, `region` STRING, `currency` STRING, `order_status` STRING, `discount_amount` STRING, `declared_total` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/orders/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_retail_order_items (`participant_key` STRING, `source_row_id` STRING, `order_id` STRING, `line_number` STRING, `product_id` STRING, `quantity` STRING, `unit_price` STRING, `discount_amount` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/order_items/'""")


## Expected result

Four Landing CSV tables and four Bronze Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
